#  pretrain
- megatron/training/training.py
1. 初始化megatron
2. 使用model_provider设置模型、优化器和lr计划
3. 调用train_val_test_data_provider获取train/val/test数据集
4. 使用forward_step_func训练模型

## pretrain 调用initialize_megatron
- megatron/training/initialize.py

它的主要职责是为大规模语言模型的训练搭建基础环境，包括解析参数、设置全局变量、初始化 PyTorch 分布式通信（如数据并行、张量并行、流水线并行）以及配置随机种子等。

为了让你更清晰地理解，我们可以将这个函数的执行流程拆解为以下几个核心阶段：

## 1. 基础环境检查与参数解析
- CUDA 检查：函数首先会检查 allow_no_cuda 参数。默认情况下，Megatron 必须依赖 CUDA 进行 GPU 训练，因此会通过 torch.cuda.is_available() 确保 GPU 环境可用。
- 参数处理：它会解析传入的命令行参数（extra_args_provider）或默认参数（args_defaults）。如果传入了 parsed_args，则直接使用。
- 检查点（Checkpoint）与配置校验：
    - 如果涉及检查点格式转换（ckpt_convert_format），会进行相关的断言检查。
    - 如果设置了 use_checkpoint_args，会从已保存的检查点中加载之前的训练参数，确保训练配置的一致性。
    - 支持通过 YAML 配置文件（yaml_cfg）来验证和覆盖参数。
    
## 2. 设置全局变量与辅助组件
- 全局变量初始化：调用 set_global_variables(args)，将解析好的参数设为全局可访问，并同时构建分词器（Tokenizer）、初始化计时器（Timers）和 TensorBoard 记录器等。
- 日志与异步保存：设置日志级别。如果开启了异步保存且使用了持久化工作进程，会初始化相关的后台进程。
- <font color='red'>重跑（Rerun）状态机初始化</font>：为了支持训练任务的容错和重跑，函数会定义状态的保存与恢复函数（主要涉及随机数生成器的状态），并初始化重跑状态机。
- 批次不变模式：如果开启了 batch_invariant_mode，会全局启用该模式（<font color='red'>通常用于调试或确保确定性训练</font>）。


## 3. 分布式环境初始化（核心逻辑）
这部分是函数的核心，<font color='red'>主要通过内部定义的 finish_mpu_init() 函数来完成</font>：

- <b>_initialize_distributed</b>：这是最关键的一步。它会<font color='red'>调用 PyTorch 的 torch.distributed.init_process_group 来建立进程间的通信后端（如 NCCL），并根据配置划分数据并行（DP）、张量并行（TP）和流水线并行（PP）的进程组。</font>
- 设置随机种子：调用 _set_random_seed 确保不同进程间的随机数生成是同步且可复现的，这对模型训练的确定性至关重要。
- MoE 辅助损失初始化：如果模型使用了混合专家（MoE）结构，会在此处初始化相关的辅助损失缩放值。


## 4. 初始化策略与后续配置
函数根据 skip_mpu_initialization 和 lazy_mpu_init 参数，提供了三种不同的初始化路径：
1. <b>跳过初始化</b>：如果 skip_mpu_initialization=True，直接返回 None。
2. <b>延迟初始化（Lazy Init）</b>：<font color='red'>如果 args.lazy_mpu_init=True，函数只会设置基础的张量并行全局变量，并返回 finish_mpu_init 函数。这意味着分布式环境的完全初始化将被推迟，交给外部的 DDP（分布式数据并行）管理器在合适的时机调用。</font>
3. <b>立即初始化（默认路径）</b>：
    - <font color='red'>直接执行 finish_mpu_init() 完成所有分布式环境的搭建。</font>
    - 自动恢复（Autoresume）：<font color='red'>初始化自动恢复功能，以便训练意外中断后能从断点继续。</font>
    - <font color='red'>编译依赖：调用 _compile_dependencies() 编译并加载自定义的 CUDA 融合算子（Fused Kernels），以提升训练效率。</font>
    - TP 通信重叠：<font color='red'>如果开启了 tp_comm_overlap，会初始化相关的通信器以优化张量并行时的通信与计算重叠。</font>
    
总结来说，initialize_megatron 就像是 Megatron-LM 训练开始前的“总指挥”，它确保了所有的 GPU 进程能够正确组队、顺畅通信，并且所有的训练配置、随机状态和底层优化算子都已准备就绪。

In [ ]:
"""Megatron initialization."""
import logging
import os
import random
import time
import warnings
from datetime import timedelta

import numpy as np
import torch

from megatron.core import mpu, tensor_parallel
from megatron.core.fusions.fused_bias_dropout import bias_dropout_add_fused_train
from megatron.core.fusions.fused_bias_gelu import bias_gelu
from megatron.core.fusions.fused_bias_swiglu import bias_swiglu
from megatron.core.parallel_state import create_group
from megatron.core.rerun_state_machine import (
    RerunDiagnostic,
    RerunErrorInjector,
    RerunMode,
    initialize_rerun_state_machine,
)
from megatron.core.transformer.custom_layers.batch_invariant_kernels import enable_batch_invariant_mode
from megatron.core.utils import get_te_version, is_te_min_version, is_torch_min_version



from megatron.training import get_adlr_autoresume, get_args, get_tensorboard_writer
from megatron.training.utils import print_rank_0, warn_rank_0
from megatron.training import inprocess_restart
from megatron.training.arguments import parse_args, validate_args
from megatron.training.async_utils import init_persistent_async_worker
from megatron.training.checkpointing import load_args_from_checkpoint
from megatron.training.global_vars import set_global_variables
from megatron.training.utils import is_rank0
from megatron.training.yaml_arguments import validate_yaml

logger = logging.getLogger(__name__)


def initialize_megatron(
    extra_args_provider=None,
    args_defaults={},
    ignore_unknown_args=False,
    allow_no_cuda=False,
    skip_mpu_initialization=False,
    get_embedding_ranks=None,
    get_position_embedding_ranks=None,
    parsed_args=None,
    store=None,
):
    """Set global variables, initialize distributed, and
    set autoresume and random seeds.
    `allow_no_cuda` should not be set unless using megatron for cpu only
    data processing. In general this arg should not be set unless you know
    what you are doing.
    Returns a function to finalize distributed env initialization
    (optionally, only when args.lazy_mpu_init == True)
    """
    if not allow_no_cuda:
        # Make sure cuda is available.
        assert torch.cuda.is_available(), "Megatron requires CUDA."

    # Parse arguments
    if parsed_args is None:
        args = parse_args(extra_args_provider, ignore_unknown_args)
    else:
        args = parsed_args

    # Prep for checkpoint conversion.
    if args.ckpt_convert_format is not None:
        assert args.ckpt_convert_save is not None
        assert args.load is not None
        args.exit_on_missing_checkpoint = True

    if args.use_checkpoint_args or args_defaults.get("use_checkpoint_args", False):
        assert args.load is not None or args.pretrained_checkpoint is not None, "--use-checkpoint-args requires --load or --pretrained-checkpoint argument"
        assert args.non_persistent_ckpt_type != "local", (
            "--use-checkpoint-args is not supported with --non_persistent_ckpt_type=local. "
            "Two-stage checkpoint loading is not implemented, and all arguments must be defined "
            "before initializing LocalCheckpointManager."
        )
        load_args_from_checkpoint(args, load_arg='pretrained_checkpoint')
        load_args_from_checkpoint(args)

    if args.yaml_cfg is not None:
        args = validate_yaml(args, args_defaults)
    else:
        validate_args(args, args_defaults)

    # set global args, build tokenizer, and set adlr-autoresume,
    # tensorboard-writer, and timers.
    set_global_variables(args)

    # set logging level
    setup_logging()

    if args.async_save and args.use_persistent_ckpt_worker:
        init_persistent_async_worker(args.rank, 'forkserver')

    # init rerun state
    def state_save_func():
        return {'rng_tracker_states': tensor_parallel.get_cuda_rng_tracker().get_states()}

    def state_restore_func(state_dict):
        if state_dict['rng_tracker_states']:
            tensor_parallel.get_cuda_rng_tracker().set_states(state_dict['rng_tracker_states'])

    args = get_args()
    initialize_rerun_state_machine(
        state_save_func=state_save_func,
        state_restore_func=state_restore_func,
        mode=RerunMode(args.rerun_mode),
        error_injector=RerunErrorInjector(
            error_injection_rate=args.error_injection_rate,
            error_injection_type=RerunDiagnostic(args.error_injection_type),
        ),
        result_rejected_tracker_filename=args.result_rejected_tracker_filename,
    )
    
    if args.batch_invariant_mode:
        print_rank_0("Enabling batch invariant mode globally")
        enable_batch_invariant_mode()

    # torch.distributed initialization
    def finish_mpu_init():
        args = get_args()
        '''
        Pytorch distributed.
        
        传递嵌入层参数：
        代码中传入的 get_embedding_ranks 和 get_position_embedding_ranks 是非常关键的参数。
        在大模型的流水线并行（PP）中，通常只有流水线的第一个阶段（负责输入）和最后一个阶段（负责输出）才需要 Embedding（词嵌入）层。
        这两个参数就是告诉框架：“请把负责首尾 Embedding 的 GPU 单独拉一个群”，以便它们之间能同步权重。
        ''' 
        _initialize_distributed(get_embedding_ranks, get_position_embedding_ranks, store)

        # Random seeds for reproducibility.
        print_rank_0("> setting random seeds to {} ...".format(args.seed))
        '''
        确保跨卡训练的可复现性 (_set_random_seed)
        
        在分布式训练中，随机性是一个极易被忽视但影响巨大的因素。如果每个 GPU 的随机种子完全一样或完全混乱，
        会导致模型训练出现严重问题（例如所有数据并行副本学到了一模一样的东西，或者 Dropout 失效）。
        
        差异化种子设置：_set_random_seed 的作用不仅仅是设一个数。
        它会根据当前 GPU 的 rank（编号）和 data_parallel_rank（数据并行组编号）对基础种子（args.seed）进行偏移计算。
        
        '''
        _set_random_seed(
            args.seed,
            args.data_parallel_random_init,
            args.te_rng_tracker,
            args.inference_rng_tracker,
            use_cudagraphable_rng=args.cuda_graph_impl != "none",
        )

        # Setup MoE aux loss scale value.
        if args.num_experts is not None:
            from megatron.core.transformer.moe.router import MoEAuxLossAutoScaler
            '''
            自动调节 Loss 权重：如果你的模型开启了专家网络（args.num_experts is not None），代码会引入 MoEAuxLossAutoScaler。
            MoE 模型在训练时，为了让不同的“专家”都能被均匀地用到，通常会引入一个辅助损失函数（Auxiliary Loss）。
            '''
            MoEAuxLossAutoScaler.set_loss_scale(torch.ones(1, device=torch.cuda.current_device()))

    if skip_mpu_initialization:
        return None

    args = get_args()
    if args.lazy_mpu_init:
        # TODO is this still a necessary option?
        args.use_cpu_initialization = True
        # delayed initialization of DDP-related stuff
        # We only set basic DDP globals
        mpu.set_tensor_model_parallel_world_size(args.tensor_model_parallel_size)
        # and return function for external DDP manager
        # to call when it has DDP initialized
        mpu.set_tensor_model_parallel_rank(args.rank)
        
        return finish_mpu_init
    else:
        # Megatron's MPU is the master. Complete initialization right away.
        finish_mpu_init()

        # Autoresume.
        _init_autoresume()

        # Compile dependencies.
        _compile_dependencies()

        if args.tp_comm_overlap:
            # TODO: Should this be activated with just decoder-tp-comm-overlap too?
            _initialize_tp_communicators()

        # No continuation function
        return None


## _initialize_distributed

它的核心职责可以概括为两步：
- 首先建立底层的 GPU 通信网络（让所有显卡“连上网”），
- 然后在此基础上划分具体的并行策略小组（明确每张卡在模型中“扮演什么角色”）。

In [ ]:
from megatron.core import mpu, tensor_parallel


def _initialize_distributed(get_embedding_ranks, get_position_embedding_ranks, store):
    """Initialize torch.distributed and core model parallel."""
    args = get_args()

    device_count = torch.cuda.device_count()
    
    '''
    防重入：如果已经初始化，直接跳过后续步骤，避免重复建立连接导致报错。
    '''
    if torch.distributed.is_initialized():

        print_rank_0("torch distributed is already initialized, skipping initialization ...")
        args.rank = torch.distributed.get_rank()
        args.world_size = torch.distributed.get_world_size()

    else:

        '''
        基础设备绑定:
        
        如果是首次初始化，它会根据当前进程的 local_rank（本地编号），
        通过 torch.cuda.set_device 将当前进程“绑定”到指定的物理 GPU 上。
        
        
        '''
        print_rank_0("> initializing torch distributed ...")
        # Manually set the device ids.
        if device_count > 0:
            torch.cuda.set_device(args.local_rank)
            device_id = torch.device(f'cuda:{args.local_rank}')
        else:
            device_id = None

        '''
        如果你开启了 CUDA Graph（一种极致的 GPU 计算加速技术），
        它会强制将当前进程切换到非默认的 CUDA Stream（计算流）上，
        这是为了满足 CUDA Graph 录制的底层硬件要求。
        '''
        # Set to non-default stream for cudagraph capturing.
        if args.cuda_graph_impl == "transformer_engine":
            torch.cuda.set_stream(torch.cuda.Stream())

        # Set flight recorder env vars if specified.
        # Priority: pre-existing environment variable > MLM argument.
        # All vars follow the same setdefault semantics: if already set in the
        # environment we warn and keep the user's value; otherwise we apply the
        # value derived from the MLM argument / flag.
        # The block is also triggered when either path env var is already set
        # so that the remaining defaults are applied consistently.
        '''
        “飞行记录仪”是 NVIDIA 针对大规模分布式训练提供的一套底层通信调试与诊断工具。
        当训练出现莫名其妙的卡死（Hang）或 NCCL 通信超时时，它能记录下详细的通信轨迹。
        '''
        _fr_path = (
            args.flight_recorder_dump_path
            or os.environ.get('TORCH_FR_DUMP_TEMP_FILE')
            or os.environ.get('TORCH_NCCL_DEBUG_INFO_TEMP_FILE')
        )
        '''
        这段逻辑在优雅地处理环境变量:
        它会优先检查你系统里是否已经设置了相关的环境变量（如 TORCH_FR_DUMP_TEMP_FILE）。
        如果没有，它会根据你传入的参数（如 args.flight_recorder_dump_path）自动生成一套默认配置并写入环境变量，确保后续如果通信出问题，系统能自动留下“黑匣子”数据供排查。
        '''
        if _fr_path is not None:
            _fr_dump_prefix = _fr_path
            if os.path.isdir(_fr_path):
                _fr_dump_prefix = os.path.join(_fr_path, '_dump_')
                warn_rank_0(
                    "Flight recorder: using directory "
                    f"'{_fr_path}' for dump path, appending per-rank prefix "
                    f"'{_fr_dump_prefix}'."
                )
            _fr_env_defaults = {
                'TORCH_FR_DUMP_TEMP_FILE': _fr_dump_prefix,
                'TORCH_NCCL_DEBUG_INFO_TEMP_FILE': _fr_dump_prefix,
                'TORCH_NCCL_TRACE_BUFFER_SIZE': str(args.flight_recorder_trace_buffer_size),
                'TORCH_NCCL_DUMP_ON_TIMEOUT': str(int(args.flight_recorder_dump_on_timeout)),
                'TORCH_INCLUDE_STACK_TRACE': str(int(args.flight_recorder_include_stack_trace)),
                'TORCH_INCLUDE_ONLY_ACTIVE': str(int(args.flight_recorder_include_only_active)),
                'TORCH_NCCL_EXTRA_DUMP_ON_EXEC': str(int(args.flight_recorder_extra_dump_on_exec)),
            }
            for _var, _default in _fr_env_defaults.items():
                if _var in os.environ:
                    warn_rank_0(
                        f"Flight recorder: environment variable {_var} is already set to "
                        f"'{os.environ[_var]}'; ignoring config value '{_default}'."
                    )
                else:
                    os.environ[_var] = _default
            print_rank_0(
                "Flight recorder env vars:\n"
                + "\n".join(f"  {k}={os.environ[k]}" for k in _fr_env_defaults)
            )

        '''
        参数打包：
        代码将通信后端（backend，通常是 nccl）、
        世界大小（world_size，总卡数）、当前进程编号（rank）以及一个 store 对象打包成字典。
        '''
        
        '''
        Store 的作用：
        store 是一个轻量级的键值对共享存储，用于在主节点协调所有进程，
        确保大家能在同一时刻完成连接建立。
        '''
        # Call the init process
        init_process_group_kwargs = {
            'backend': args.distributed_backend,
            'store': store,
            'world_size': args.world_size,
            'rank': args.rank,
            'timeout': timedelta(minutes=args.distributed_timeout_minutes),
        }
        '''
        Fake Process Group（调试模式）：
        代码中有一个 if args.fake_process_group 的判断。
        如果开启这个模式，它会使用 PyTorch 内置的 FakeStore 和 fake 后端。
        这意味着不建立真实的 GPU 通信，专门用于在没有多卡硬件的环境下调试代码逻辑。
        '''
        if args.fake_process_group:
            assert is_torch_min_version("2.3.0"), "Fake process group is only supported with PyTorch 2.3.0 and above."
            from torch.testing._internal.distributed.fake_pg import FakeStore
            
            store = FakeStore()
            init_process_group_kwargs['backend'] = 'fake'
            init_process_group_kwargs['store'] = store

        '''
        建立全局通信组:
        所有 GPU 进程在此刻正式联网，形成一个大集群。
        '''
        torch.distributed.init_process_group(**init_process_group_kwargs)
        inprocess_restart.maybe_force_nccl_backend_init(device_id)

    # Set the tensor model-parallel, pipeline model-parallel, and
    # data-parallel communicators.
    if device_count > 0:
        if mpu.model_parallel_is_initialized():
            print("model parallel is already initialized")
        else:
            '''
            划分具体的并行策略小组 (mpu.initialize_model_parallel)
            '''
            mpu.initialize_model_parallel(
                args.tensor_model_parallel_size,
                args.pipeline_model_parallel_size,
                args.virtual_pipeline_model_parallel_size,
                pipeline_model_parallel_comm_backend=args.pipeline_model_parallel_comm_backend,
                use_sharp=args.use_sharp,
                context_parallel_size=args.context_parallel_size,
                hierarchical_context_parallel_sizes=args.hierarchical_context_parallel_sizes,
                hybrid_context_parallel=args.hybrid_context_parallel,
                expert_model_parallel_size=args.expert_model_parallel_size,
                num_distributed_optimizer_instances=args.num_distributed_optimizer_instances,
                expert_tensor_parallel_size=args.expert_tensor_parallel_size,
                distributed_timeout_minutes=args.distributed_timeout_minutes,
                nccl_communicator_config_path=args.nccl_communicator_config_path,
                order='tp-cp-ep-dp-pp' if not args.use_tp_pp_dp_mapping else 'tp-cp-ep-pp-dp',
                get_embedding_ranks=get_embedding_ranks,
                get_position_embedding_ranks=get_position_embedding_ranks,
                create_gloo_process_groups=args.use_gloo_process_groups,
                high_priority_stream_groups=args.high_priority_stream_groups,
                sharp_enabled_group=args.sharp_enabled_group,
                create_all_gather_group=args.create_all_gather_group,
            )
            print_rank_0(
                f"> initialized tensor model parallel with size "
                f"{mpu.get_tensor_model_parallel_world_size()}"
            )
            print_rank_0(
                f"> initialized pipeline model parallel with size "
                f"{mpu.get_pipeline_model_parallel_world_size()}"
            )


# mpu.initialize_model_parallel
- 详情见mpu ../mpu.ipynb
- mpu
   - <font color='green'>mpu 实际上是指向 megatron.core.parallel_state 模块的一个别名</font>
   - <font color='red'>在底层的 torch.distributed 让所有 GPU 连上网之后，mpu 负责根据你设定的策略，给每张 GPU 划分具体的“管辖区域”和“沟通渠道”。</font>

In [ ]:
             mpu.initialize_model_parallel(
                args.tensor_model_parallel_size,
                args.pipeline_model_parallel_size,
                args.virtual_pipeline_model_parallel_size,
                pipeline_model_parallel_comm_backend=args.pipeline_model_parallel_comm_backend,
                use_sharp=args.use_sharp,
                context_parallel_size=args.context_parallel_size,
                hierarchical_context_parallel_sizes=args.hierarchical_context_parallel_sizes,
                hybrid_context_parallel=args.hybrid_context_parallel,
                expert_model_parallel_size=args.expert_model_parallel_size,
                num_distributed_optimizer_instances=args.num_distributed_optimizer_instances,
                expert_tensor_parallel_size=args.expert_tensor_parallel_size,
                distributed_timeout_minutes=args.distributed_timeout_minutes,
                nccl_communicator_config_path=args.nccl_communicator_config_path,
                order='tp-cp-ep-dp-pp' if not args.use_tp_pp_dp_mapping else 'tp-cp-ep-pp-dp',
                get_embedding_ranks=get_embedding_ranks,
                get_position_embedding_ranks=get_position_embedding_ranks,
                create_gloo_process_groups=args.use_gloo_process_groups,
                high_priority_stream_groups=args.high_priority_stream_groups,
                sharp_enabled_group=args.sharp_enabled_group,
                create_all_gather_group=args.create_all_gather_group,
            )